# 20.13 交給別人使用時，要一起帶走哪些東西？


你把修改便條交給同學，卻沒說對應哪本書、哪個版本，他可能把內容正確的便條貼錯地方。模型交付同樣需要配套：底座、文字與圖片處理工具、語音辨識器，以及選版實際採用的設定。採用LoRA時才一起交付那份修正；保留底座時，能力卡也要明寫未開啟修正。

前置是[19.11的推論與續訓檔](https://birdhackor.github.io/tiny-perceptron-vlm/19.11.html)和[20.3的模型分工](https://birdhackor.github.io/tiny-perceptron-vlm/20.3.html)。版本像書籍版次，SHA-256則像內容指紋：把同一串bytes交給它，得到同一個摘要，下載後可核對是否拿到指定內容。這件事保證的是檔案配對，不是回答品質。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
from hashlib import sha256

expected = b"base A, adapter B"
received = b"base A, adapter C"
print("期待內容指紋", sha256(expected).hexdigest()[:12])
print("收到內容指紋", sha256(received).hexdigest()[:12])
print("內容符合清單", sha256(expected).digest() == sha256(received).digest())

`b`表示bytes，這兩串是人工文字，不是真模型檔。最後是False，因為收到的調整版本不同。前12個字元只方便顯示；正式下載要核對完整指紋與檔案大小。配套清單應寫明是哪一份底座、處理工具與語音權重，讓同學依照清單取得同一套配置。

本版選定的是Qwen3-VL-2B-Instruct底座，加上Whisper-large-v3-turbo語音辨識器，沒有開啟LoRA修正。[20.8的驗證](https://birdhackor.github.io/tiny-perceptron-vlm/20.8.html)先決定這個配置，下面的最後考卷只驗收它，不再拿考卷重新挑版本。底座有2,127,532,032個參數，辨識器另有808,878,080個參數；兩份權重各自負責不同工作，不能把辨識器的結果算成圖文模型的微調成果。固定版本可回查[模型執行核對](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/natural-assistant/evidence/v4-runtime/evaluate-37221188153/selected-final-test-audit-summary.json)與[語音選版設定](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/natural-assistant/v4/asr-selection.json)。

接著要一起交出一張能力卡。它像家電的用途說明：能洗衣，不代表能烘乾；能看見招牌有字，也不代表能逐字抄對。這張卡把不同用途分開，寫出考了什麼、完成幾題，以及還沒驗收什麼。表中的「通過」同時要求內容符合事先判準、遵守提問，而且正常產生結束符號。照片與聊天依逐題判準檢查，逐字抄寫則使用固定比對規約；回答只是結束了，並不等於回答正確。

最後考卷共產生178份回答，其中171份正常結束，7份達到384個新token上限而截斷，全部在文字聊天組。以下仍把這7份算在13題的分母裡；不能只挑說完的回答計分。素材留作本課最後驗收，未參與本課微調或選版，但不能保證上游預訓練從未見過它們。

| 用途 | 最後考卷結果 | 考了什麼，結果支持到哪裡 |
| --- | --- | --- |
| 自然照片的主要場景描述 | 25/42題通過 | 42張照片，各要求一至兩句繁中描述主要內容；不是把所有細節逐一列完。 |
| 自然照片的可見事實問答 | 58/84題通過 | 同42張照片各問兩題，包含物件、屬性、數量、動作與關係。當中動作題3/11、關係題21/29，已包含在84題裡。 |
| 有沒有清楚可見的中文字 | 18/18題通過 | 18張真實照片，10張有、8張沒有；只判有無，不代表讀出文字。 |
| 真實照片裡指定區域的逐字抄寫 | 8/10題通過 | 10張照片，保留原來的繁簡字形與標點，按單區題規約處理空白；不是整張照片所有文字的轉寫。 |
| 真實文字的指定讀序與分行 | 1/3題通過 | 三張照片，依問題選定文字、排列順序並保留要求的換行；文字與順序一起正確才算通過。 |
| 中文文字請求與聊天 | 2/13題通過 | 13題裡只有1題必須依前文作答，該題未通過；其中7題截斷。沒有驗收長多輪聊天。 |
| 真人語音轉寫 | 原樣CER 112/674＝16.62%；規約CER 96/661＝14.52% | 22段普通話真人朗讀：18段FLEURS句子、4段AISHELL問句；這是聽寫字元誤差率，不是聊天成功率。 |
| 同一語音問句，先給正確文字再回答 | 2/4題通過 | 把四段AISHELL問句的來源逐字稿送進聊天核心，檢查它收到正確請求時的回答。 |
| 同一語音問句，先辨識錄音再回答 | 2/4題通過 | 使用同四段錄音的實際ASR逐字稿，接進相同聊天核心，檢查整條語音問答路線。 |

照片的42個描述和84個問答共用42張圖；有無字、單區抄寫與分行題也共用部分照片。語音的文字路與錄音路則共用四個問題。因此這些題目不是178個彼此獨立的新情境，也不適合合成一個「什麼都會」的總分。完整計分可回查[逐題結果](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/natural-assistant/evidence/v4-runtime/evaluate-37221188153/blind-review/scored/scores.json)；動作、關係和依前文題的分類見[題型子集](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/natural-assistant/evidence/v4-runtime/evaluate-37221188153/blind-review/descriptive-subsets.json)。

CER的分子是插入、刪除與替換的總次數，分母是來源逐字稿的字元數，不是「幾段錄音沒聽懂」。原樣計分保留空白與標點；規約計分先做Unicode NFKC字元形式正規化，例如把全形數字換成半形，再移除空白，仍保留正規化後的標點、大小寫與繁簡差異。18段FLEURS句子的原樣誤差是112/632，規約誤差是96/619；四段AISHELL問句兩種計分都是0/42。四個問題全都聽對，兩條聊天路線卻都只有2/4通過，正好說明[20.12的兩站責任](https://birdhackor.github.io/tiny-perceptron-vlm/20.12.html)：聽寫正確之後，助手仍要完成回答。

這張卡支持一些自然照片描述與中文讀字能力，尚不足以把成品當成可靠的通用聊天助手。動作與關係子集的差異也提醒我們，只報物件名稱不能代替看懂發生什麼，不能用一個看圖總分替它們背書。這份最後考卷沒有合成字卡；三題指定讀序也沒有驗收任意整頁文件的版面理解。語音素材是朗讀，不是帶噪聲的即興長對話。未測的用途保留未驗收，學生才知道下一份考卷應補在哪裡。

本版的[公開成品清單](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/natural-assistant/v4/public-release.json)與配置文件已完成匿名下載核對，選定配置也已在Linux CPU上以實際Chromium瀏覽器核對文字、照片、語音及清除對話。這份操作檢查使用已有的官方模型快取，沒有重新評分回答；上表仍只採最後考卷的成績。磁碟、記憶體與等待時間的實測範圍集中在[學生操作指引](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/natural-assistant/v4/STUDENT.md)第2節，不能從一次操作成功推成最低設備需求。

試用步驟見[20.2](https://birdhackor.github.io/tiny-perceptron-vlm/20.2.html)，資料與授權見[資料說明](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/natural-assistant/v4/DATA.md)。想建立自己的候選修正，可按[訓練指引](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/natural-assistant/v4/TRAINING.md)操作，再做同樣的用途驗收。交付LoRA時，推論小包仍要配對底座；精確續訓另外需要更新器、隨機狀態與進度。本版沒有啟用LoRA，不能把一份練過的候選修正自動當成最後交付配置。

練習把`received`改成與`expected`相同，最後會變True。再回答：指紋相同能否代替能力卡？替一個尚未測過的用途設計兩三道完整任務，而不只檢查程式能不能輸出。到這裡，你已能分別交代配置、實際運行與用途驗收，也知道增加資料、替換零件或壓縮模型後，應重新檢查哪一項。
